In [ ]:
!pip install sentence-transformers faiss-cpu pandas

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving ICD MEDICAL TERMS.csv to ICD MEDICAL TERMS (1).csv


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("ICD MEDICAL TERMS.csv", header=None)

# Display first few rows
print(df.head())

      0  1      2                                                  3  \
0   A00  0   A000  Cholera due to Vibrio cholerae 01, biovar chol...   
1   A00  1   A001    Cholera due to Vibrio cholerae 01, biovar eltor   
2   A00  9   A009                               Cholera, unspecified   
3  A010  0  A0100                         Typhoid fever, unspecified   
4  A010  1  A0101                                 Typhoid meningitis   

                                                   4              5  
0  Cholera due to Vibrio cholerae 01, biovar chol...        Cholera  
1    Cholera due to Vibrio cholerae 01, biovar eltor        Cholera  
2                               Cholera, unspecified        Cholera  
3                         Typhoid fever, unspecified  Typhoid fever  
4                                 Typhoid meningitis  Typhoid fever  


In [ ]:
medical_terms = df[3].dropna().astype(str).tolist()

print(medical_terms[:10])

['Cholera due to Vibrio cholerae 01, biovar cholerae', 'Cholera due to Vibrio cholerae 01, biovar eltor', 'Cholera, unspecified', 'Typhoid fever, unspecified', 'Typhoid meningitis', 'Typhoid fever with heart involvement', 'Typhoid pneumonia', 'Typhoid arthritis', 'Typhoid osteomyelitis', 'Typhoid fever with other complications']


In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert medical terms into embeddings
embeddings = model.encode(medical_terms)

print("Embeddings created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created successfully


In [ ]:
import faiss
import numpy as np

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(np.array(embeddings))

print("FAISS index created successfully")

FAISS index created successfully


In [ ]:
json_output = {
  "patient_name": "Mohammed Arif",
  "primary_diagnosis": "Acte exacebation of COPD"
}

In [ ]:
def retrieve_medical_term(query, top_k=1):

    # Convert query into embedding
    query_embedding = model.encode([query])

    # Search FAISS
    D, I = index.search(np.array(query_embedding), top_k)

    # Retrieve matched terms
    results = []

    for idx in I[0]:
        results.append(medical_terms[idx])

    return results

In [ ]:
diagnosis = json_output["primary_diagnosis"]

retrieved_terms = retrieve_medical_term(diagnosis)

print("Original Diagnosis:")
print(diagnosis)

print("\nRetrieved Medical Match:")
print(retrieved_terms[0])

Original Diagnosis:
Acte exacebation of COPD

Retrieved Medical Match:
Chronic obstructive pulmonary disease w (acute) exacerbation


In [ ]:
json_output["validated_diagnosis"] = retrieved_terms[0]

print("\n===== VALIDATED JSON OUTPUT =====\n")

for key, value in json_output.items():
    print(f"{key} : {value}")


===== VALIDATED JSON OUTPUT =====

patient_name : Mohammed Arif
primary_diagnosis : Acte exacebation of COPD
validated_diagnosis : Chronic obstructive pulmonary disease w (acute) exacerbation


In [ ]:
json_outputs = [

{
  "patient_name": "61-year-old male",
  "primary_diagnosis": "Elevated PSA"
},

{
  "patient_name": "68-year-old Caucasian male",
  "primary_diagnosis": "Bladder cancer"
},

{
  "patient_name": "78-year-old female",
  "primary_diagnosis": "hypertension"
},

{
  "patient_name": "41-year-old white female",
  "primary_diagnosis": "Osteoarthritis"
},

{
  "patient_name": "49-year-old white female",
  "primary_diagnosis": "Acne with folliculitis"
}

]

In [ ]:
for patient in json_outputs:

    diagnosis = patient["primary_diagnosis"]

    retrieved = retrieve_medical_term(diagnosis)

    patient["validated_diagnosis"] = retrieved[0]

    print("\n========== RAG VALIDATION ==========\n")

    print(f"Patient Name         : {patient['patient_name']}")
    print(f"Original Diagnosis   : {diagnosis}")
    print(f"Validated Diagnosis  : {retrieved[0]}")

    print("\n====================================")


========== RAG VALIDATION ==========

Patient Name         : 61-year-old male
Original Diagnosis   : Elevated PSA
Validated Diagnosis  : Elevated prostate specific antigen [PSA]


========== RAG VALIDATION ==========

Patient Name         : 68-year-old Caucasian male
Original Diagnosis   : Bladder cancer
Validated Diagnosis  : Malignant neoplasm of bladder, unspecified


========== RAG VALIDATION ==========

Patient Name         : 78-year-old female
Original Diagnosis   : hypertension
Validated Diagnosis  : Postprocedural hypertension


========== RAG VALIDATION ==========

Patient Name         : 41-year-old white female
Original Diagnosis   : Osteoarthritis
Validated Diagnosis  : Osteoarthritis of knee, unspecified


========== RAG VALIDATION ==========

Patient Name         : 49-year-old white female
Original Diagnosis   : Acne with folliculitis
Validated Diagnosis  : Folliculitis decalvans



In [ ]:
secondary_json_outputs = [

    {
        "patient_name": "61-year-old male",
        "secondary_diagnosis": ["null"]
    },

    {
        "patient_name": "68-year-old Caucasian male",
        "secondary_diagnosis": ["null"]
    },

    {
        "patient_name": "78-year-old female patient",
        "secondary_diagnosis": [
            "Spinal stenosis",
            "Syncope"
        ]
    },

    {
        "patient_name": "41-year-old white female",
        "secondary_diagnosis": [
            "Chronic sacroiliitis",
            "Lumbar spondylosis",
            "Migraine",
            "TMJ arthropathy"
        ]
    },

    {
        "patient_name": "49-year-old white female",
        "secondary_diagnosis": ["null"]
    }
]

In [ ]:
print("\n========== SECONDARY DIAGNOSIS VALIDATION ==========\n")

for patient in secondary_json_outputs:

    patient_name = patient["patient_name"]

    diagnoses = patient["secondary_diagnosis"]

    print("Patient Name         :", patient_name)

    valid_found = False

    for diagnosis in diagnoses:

        if diagnosis.lower() == "null":
            continue

        valid_found = True

        validated_term = retrieve_medical_term(diagnosis)[0]

        print("Original Diagnosis   :", diagnosis)
        print("Validated Diagnosis  :", validated_term)

        print()

    if not valid_found:

        print("No Secondary Diagnosis Found")

    print("\n====================================\n")


========== SECONDARY DIAGNOSIS VALIDATION ==========

Patient Name         : 61-year-old male
No Secondary Diagnosis Found


Patient Name         : 68-year-old Caucasian male
No Secondary Diagnosis Found


Patient Name         : 78-year-old female patient
Original Diagnosis   : Spinal stenosis
Validated Diagnosis  : Spinal stenosis, site unspecified

Original Diagnosis   : Syncope
Validated Diagnosis  : Syncope and collapse



Patient Name         : 41-year-old white female
Original Diagnosis   : Chronic sacroiliitis
Validated Diagnosis  : Sacroiliitis, not elsewhere classified

Original Diagnosis   : Lumbar spondylosis
Validated Diagnosis  : Other spondylosis, lumbar region

Original Diagnosis   : Migraine
Validated Diagnosis  : Migraine without aura, intractable, with status migrainosus

Original Diagnosis   : TMJ arthropathy
Validated Diagnosis  : Unspecified TMJ joint disorder, unspecified side



Patient Name         : 49-year-old white female
No Secondary Diagnosis Found




In [ ]:
from google.colab import files
uploaded = files.upload()

Saving drugsComTrain_raw.csv to drugsComTrain_raw.csv


In [ ]:
import pandas as pd

med_df = pd.read_csv("drugsComTrain_raw.csv")

In [ ]:
med_df.columns

Index(['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date',
       'usefulCount'],
      dtype='object')

In [ ]:
medicine_terms = med_df["drugName"].dropna().unique().tolist()

print(medicine_terms[:10])

['Valsartan', 'Guanfacine', 'Lybrel', 'Ortho Evra', 'Buprenorphine / naloxone', 'Cialis', 'Levonorgestrel', 'Aripiprazole', 'Keppra', 'Ethinyl estradiol / levonorgestrel']


In [ ]:
medication_json_outputs = [

    {
        "patient_name": "61-year-old male",
        "discharge_medications": [
            "Proscar"
        ]
    },

    {
        "patient_name": "68-year-old Caucasian male",
        "discharge_medications": [
            "null"
        ]
    },

    {
        "patient_name": "78-year-old female patient",
        "discharge_medications": [
            "Atenolol 50 mg daily",
            "Premarin 0.625 mg daily",
            "Calcium with vitamin D two to three pills daily",
            "Multivitamin daily",
            "Aspirin as needed",
            "TriViFlor 25 mg two pills daily"
        ]
    },

    {
        "patient_name": "41-year-old white female",
        "discharge_medications": [
            "Methadone",
            "Norco",
            "Tenormin",
            "Skelaxin",
            "Wellbutrin SR",
            "Naprosyn"
        ]
    },

    {
        "patient_name": "49-year-old white female",
        "discharge_medications": [
            "amoxicillin 500 mg b.i.d.",
            "Tazorac cream 0.1",
            "Septra DS"
        ]
    }
]

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "pritamdeka/S-PubMedBert-MS-MARCO"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
medicine_embeddings = embedding_model.encode(
    medicine_terms,
    convert_to_numpy=True
)

In [ ]:
import faiss
import numpy as np

dimension = medicine_embeddings.shape[1]

med_index = faiss.IndexFlatL2(dimension)

med_index.add(np.array(medicine_embeddings))

print("Medicine FAISS index created successfully")

Medicine FAISS index created successfully


In [ ]:
def retrieve_medicine_term(query, top_k=1):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = med_index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_terms = [
        medicine_terms[idx]
        for idx in indices[0]
    ]

    return retrieved_terms

In [ ]:
import re

def retrieve_medicine_term(query):

    query_lower = query.lower()

    # remove punctuation
    cleaned_query = re.sub(
        r'[^a-zA-Z0-9\s]',
        ' ',
        query_lower
    )

    query_words = cleaned_query.split()

    # exact medicine presence check
    for med in medicine_terms:

        med_lower = med.lower()

        if med_lower in cleaned_query:

            return [query]

    # partial word match
    for med in medicine_terms:

        med_lower = med.lower()

        med_words = med_lower.split()

        for word in med_words:

            if word in query_words:

                return [query]

    return ["NO_CLOSE_MATCH"]

In [ ]:
def retrieve_medicine_term(query, top_k=1):

    # -------------------------------------
    # EXACT MATCH FIRST
    # -------------------------------------

    for med in medicine_terms:

        if med.lower() in query.lower():

            return [med]

    # -------------------------------------
    # SEMANTIC SEARCH SECOND
    # -------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = med_index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_terms = [
        medicine_terms[idx]
        for idx in indices[0]
    ]

    return retrieved_terms

In [ ]:
print("\n========== MEDICATION VALIDATION ==========\n")

for patient in medication_json_outputs:

    patient_name = patient["patient_name"]

    medications = patient["discharge_medications"]

    print("Patient Name         :", patient_name)

    valid_found = False

    for medicine in medications:

        if medicine.lower() == "null":
            continue

        valid_found = True

        validated_medicine = retrieve_medicine_term(medicine)[0]

        print("Original Medication  :", medicine)
        print("Validated Medication :", validated_medicine)

        print()

    if not valid_found:

        print("No Discharge Medications Found")

    print("\n====================================\n")


========== MEDICATION VALIDATION ==========

Patient Name         : 61-year-old male
Original Medication  : Proscar
Validated Medication : Proscar



Patient Name         : 68-year-old Caucasian male
No Discharge Medications Found


Patient Name         : 78-year-old female patient
Original Medication  : Atenolol 50 mg daily
Validated Medication : Atenolol 50 mg daily

Original Medication  : Premarin 0.625 mg daily
Validated Medication : Premarin 0.625 mg daily

Original Medication  : Calcium with vitamin D two to three pills daily
Validated Medication : Calcium with vitamin D two to three pills daily

Original Medication  : Multivitamin daily
Validated Medication : Multivitamin daily

Original Medication  : Aspirin as needed
Validated Medication : Aspirin as needed

Original Medication  : TriViFlor 25 mg two pills daily
Validated Medication : TriViFlor 25 mg two pills daily



Patient Name         : 41-year-old white female
Original Medication  : Methadone
Validated Medication : Meth

In [ ]:
secondary_results = []

for patient in secondary_json_outputs:

    patient_name = patient["patient_name"]

    diagnoses = patient["secondary_diagnosis"]

    for diagnosis in diagnoses:

        if diagnosis.lower() == "null":
            continue

        validated_term = retrieve_medical_term(diagnosis)[0]

        secondary_results.append({

            "patient_name": patient_name,

            "original": diagnosis,

            "validated": validated_term
        })

In [ ]:
medication_results = []

for patient in medication_json_outputs:

    patient_name = patient["patient_name"]

    medications = patient["discharge_medications"]

    for medicine in medications:

        if medicine.lower() == "null":
            continue

        validated_medicine = retrieve_medicine_term(medicine)[0]

        medication_results.append({

            "patient_name": patient_name,

            "original": medicine,

            "validated": validated_medicine
        })

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.7 MB/s eta 0:00:00


In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak
)

from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

styles = getSampleStyleSheet()

pdf = SimpleDocTemplate(
    "RAG_Validation_Report.pdf",
    pagesize=letter
)

content = []

# ======================================================
# PRIMARY DIAGNOSIS
# ======================================================

content.append(
    Paragraph(
        "<b>PRIMARY DIAGNOSIS VALIDATION</b>",
        styles['Title']
    )
)

content.append(Spacer(1, 12))

for item in json_outputs:

    text = f"""
    <b>Patient Name:</b> {item['patient_name']}<br/>
    <b>Original Diagnosis:</b> {item['primary_diagnosis']}<br/>
    <b>Validated Diagnosis:</b> {item['validated_diagnosis']}<br/><br/>
    """

    content.append(
        Paragraph(
            text,
            styles['BodyText']
        )
    )

    content.append(
        Spacer(1, 12)
    )

content.append(PageBreak())

# ======================================================
# SECONDARY DIAGNOSIS
# ======================================================

content.append(
    Paragraph(
        "<b>SECONDARY DIAGNOSIS VALIDATION</b>",
        styles['Title']
    )
)

content.append(Spacer(1, 12))

for item in secondary_results:

    text = f"""
    <b>Patient Name:</b> {item['patient_name']}<br/>
    <b>Original Diagnosis:</b> {item['original']}<br/>
    <b>Validated Diagnosis:</b> {item['validated']}<br/><br/>
    """

    content.append(
        Paragraph(
            text,
            styles['BodyText']
        )
    )

    content.append(
        Spacer(1, 12)
    )

content.append(PageBreak())

# ======================================================
# MEDICATION VALIDATION
# ======================================================

content.append(
    Paragraph(
        "<b>MEDICATION VALIDATION</b>",
        styles['Title']
    )
)

content.append(Spacer(1, 12))

for item in medication_results:

    text = f"""
    <b>Patient Name:</b> {item['patient_name']}<br/>
    <b>Original Medication:</b> {item['original']}<br/>
    <b>Validated Medication:</b> {item['validated']}<br/><br/>
    """

    content.append(
        Paragraph(
            text,
            styles['BodyText']
        )
    )

    content.append(
        Spacer(1, 12)
    )

pdf.build(content)

print("PDF Generated Successfully")

PDF Generated Successfully
